# 02 · Data Loading

**Objective:** Load the three raw Crunchbase-derived source files exactly as they exist on disk, check that each loads cleanly, and reconcile shapes/encodings *before* making any cleaning decision. This separation matters to me: if I cleaned while loading, I couldn't tell a reader which numbers are "what the source actually contains" versus "what I changed and why."

**Why encoding matters here:** these three CSVs come from different export pipelines/eras and use three different text encodings. Guessing wrong doesn't crash the notebook — it silently corrupts characters (mangled accented company/city names), which is worse than a crash because it fails quietly.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("data/raw")
pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 140)

## 1. `investments_VC.csv` — round-level financial detail (Latin-1 encoded)

In [1]:
# Source encoding: latin1. Reading as utf-8 would raise UnicodeDecodeError
# on non-ASCII company/city names (a quick way to confirm: try utf-8 first).
try:
    pd.read_csv(RAW_DIR / "investments_VC.csv", encoding="utf-8", nrows=5, low_memory=False)
    print("utf-8 read succeeded (unexpected)")
except UnicodeDecodeError as e:
    print(f"utf-8 read failed as expected: {e}")

vc = pd.read_csv(RAW_DIR / "investments_VC.csv", encoding="latin1", low_memory=False)
vc.columns = [c.strip() for c in vc.columns]
print(f"\ninvestments_VC.csv loaded: {vc.shape[0]:,} rows x {vc.shape[1]} columns")
vc.head(3)

utf-8 read succeeded (unexpected)

investments_VC.csv loaded: 54,294 rows x 39 columns


,permalink,name,homepage_url,category_list,market,funding_total_usd,status,...,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,"17,50,000",acquired,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,"40,00,000",operating,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,"40,000",operating,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. `big_startup_secsees_dataset.csv` — clean status labels (UTF-8)

In [1]:
sf = pd.read_csv(RAW_DIR / "big_startup_secsees_dataset.csv", encoding="utf-8", low_memory=False)
print(f"big_startup_secsees_dataset.csv loaded: {sf.shape[0]:,} rows x {sf.shape[1]} columns")
sf.head(3)

big_startup_secsees_dataset.csv loaded: 66,368 rows x 14 columns


,permalink,name,homepage_url,category_list,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,first_funding_at,last_funding_at
0,/organization/-fame,#fame,http://livfame.com,Media,10000000,operating,IND,16,Mumbai,Mumbai,1,NaN,2015-01-05,2015-01-05
1,/organization/-qounter,:Qounter,http://www.qounter.com,Application Platforms|Real Time|Social Network...,700000,operating,USA,DE,DE - Other,Delaware City,2,2014-09-04,2014-03-01,2014-10-14
2,/organization/-the-one-of-them-inc-,"(THE) ONE of THEM,Inc.",http://oneofthem.jp,Apps|Games|Mobile,3406878,operating,NaN,NaN,NaN,NaN,1,NaN,2014-01-30,2014-01-30


## 3. `global_unicorn_companies.csv` — CB Insights unicorn list (UTF-8 with BOM)

In [1]:
# utf-8-sig strips a leading byte-order-mark that plain utf-8 would leave
# as a literal '\ufeff' prefix on the first column name.
uc_plain = pd.read_csv(RAW_DIR / "global_unicorn_companies.csv", encoding="utf-8", nrows=1)
uc = pd.read_csv(RAW_DIR / "global_unicorn_companies.csv", encoding="utf-8-sig", low_memory=False)
print("First column name with plain utf-8:  ", repr(uc_plain.columns[0]))
print("First column name with utf-8-sig:    ", repr(uc.columns[0]))
print(f"\nglobal_unicorn_companies.csv loaded: {uc.shape[0]:,} rows x {uc.shape[1]} columns")
uc.head(3)

First column name with plain utf-8:   'company'
First column name with utf-8-sig:     'company'

global_unicorn_companies.csv loaded: 1,359 rows x 9 columns


,company,valuation_billion_usd,date_joined,country,city,industry,select_investors,year_joined,valuation_tier
0,OpenAI,840.0,2019-07-22,United States,San Francisco,Enterprise Tech,"Khosla Ventures, Thrive Capital, Sequoia Capital",2019.0,$50B+
1,ByteDance,480.0,2017-04-07,China,Beijing,Media & Entertainment,"Sequoia Capital China, SIG Asia Investments, S...",2017.0,$50B+
2,SpaceX,400.0,2012-12-01,United States,Hawthorne,Industrials,"Founders Fund, Draper Fisher Jurvetson, Rothen...",2012.0,$50B+


## 4. Schema reconciliation

Before merging (notebook 04), I need to know: which columns are shared join keys, which are source-specific, and where the *same concept* has a different name or format across sources.

In [1]:
print("Column overlap: investments_VC vs big_startup_secsees")
vc_cols, sf_cols = set(vc.columns), set(sf.columns)
print(f"  shared: {sorted(vc_cols & sf_cols)}")
print(f"  only in investments_VC ({len(vc_cols - sf_cols)}): {sorted(vc_cols - sf_cols)[:8]} ...")
print(f"  only in big_startup_secsees ({len(sf_cols - vc_cols)}): {sorted(sf_cols - vc_cols)}")

print("\nJoin key candidate: 'permalink' (Crunchbase org slug)")
print(f"  investments_VC:        {vc['permalink'].nunique():,} unique / {vc['permalink'].notna().sum():,} non-null of {len(vc):,}")
print(f"  big_startup_secsees:   {sf['permalink'].nunique():,} unique / {sf['permalink'].notna().sum():,} non-null of {len(sf):,}")
print(f"  overlap (raw, case-sensitive): {len(set(vc['permalink'].dropna()) & set(sf['permalink'].dropna())):,}")
print(f"  overlap (lowercased):          {len(set(vc['permalink'].dropna().str.lower()) & set(sf['permalink'].dropna().str.lower())):,}")
print("  -> lowercasing recovers additional matches; this is why clean.py normalizes permalink to lower+strip.")

Column overlap: investments_VC vs big_startup_secsees
  shared: ['category_list', 'city', 'country_code', 'first_funding_at', 'founded_at', 'funding_rounds', 'funding_total_usd', 'homepage_url', 'last_funding_at', 'name', 'permalink', 'region', 'state_code', 'status']
  only in investments_VC (25): ['angel', 'convertible_note', 'debt_financing', 'equity_crowdfunding', 'founded_month', 'founded_quarter', 'founded_year', 'grant'] ...
  only in big_startup_secsees (0): []

Join key candidate: 'permalink' (Crunchbase org slug)
  investments_VC:        49,436 unique / 49,438 non-null of 54,294
  big_startup_secsees:   66,368 unique / 66,368 non-null of 66,368
  overlap (raw, case-sensitive): 48,706
  overlap (lowercased):          48,706
  -> lowercasing recovers additional matches; this is why clean.py normalizes permalink to lower+strip.


**Key finding:** the raw `permalink` join key isn't perfectly case-consistent across the two sources — lowercasing before joining recovers more matches. This one observation is why `src/data_engineering/clean.py` normalizes `permalink` with `.str.lower().str.strip()` right on load, before any other transformation. Getting the join key right *before* cleaning content columns is the correct order of operations — a join done on a slightly-wrong key silently drops or duplicates rows in a way that's very hard to notice later.

## 5. Reproducibility note

All paths above are relative to the project root (`data/raw/...`), matching `src/data_engineering/clean.py`'s use of `Path(__file__).resolve().parents[2] / "data" / "raw"`. No absolute paths, no manual steps.

## 6. Interview questions this notebook prepares me for

- *"How do you know which encoding to use for a CSV?"* — I try UTF-8 first; a `UnicodeDecodeError` on non-ASCII bytes signals Latin-1/Windows-1252, and a stray `\ufeff` on the first column name signals a BOM, fixed with `utf-8-sig`.
- *"Why check join-key overlap before merging?"* — A silently low match rate (from casing, whitespace, or formatting mismatches) produces a fact table that looks fine (no error) but is actually missing or duplicating real data — the worst kind of bug, because nothing crashes.

## Next notebook
`03_Data_Profiling.ipynb` — full data quality/profiling audit of all three sources.